In [ ]:
import os
import polars as pl
from pathlib import Path

DATA_PATH = Path('/data/gusev/USERS/jpconnor/data/')
COMPASS_PATH = DATA_PATH / 'CAIA/COMPASS/'

hist_mrns = pl.read_csv(COMPASS_PATH / 'histology_mrns.csv')
old_annotation_mrns = (pl.read_csv(COMPASS_PATH / 'LLM_NEPC_labels/baca_lab_annotations.csv')
                       .with_columns(pl.lit('old_annotations').alias('annotation_version')))
new_annotation_mrns = (pl.read_csv(COMPASS_PATH / 'LLM_NEPC_labels/Oct_1_2026_baca_annotations.csv')
                       .filter(pl.col('Manual_label').is_not_null())
                       .with_columns(pl.lit('new_annotations').alias('annotation_version')))

llm_labels = pl.read_parquet('/data/gusev/USERS/jpconnor/data/LLM_annotations/LLM_primary_subtype/primary_subtype_labels.parquet')
pred_mrns = pl.read_csv(COMPASS_PATH / 'survival_analysis/prediction_inputs_adt/aggregated_landmark0.csv')

spelling_map = {'biomarker_driven' : 'biomarker', 'conventionnal' : 'conventional',
                'squamous_transformation' : 'nepc', 'neoadjuvent trial' : 'conventional',
                'progression' : 'conventional',}
cols_to_keep = ['DFCI_MRN', 'manual_annotation', 'annotation_version']

full_annotations = (pl.concat(
    [(old_annotation_mrns.rename({'simplified_manual_platinum_reason' : 'manual_annotation'}).select(cols_to_keep)),
    (new_annotation_mrns.rename({'Manual_label' : 'manual_annotation'}).select(cols_to_keep))], how='vertical')
    .with_columns(pl.col('manual_annotation').str.to_lowercase().str.strip_chars().replace(spelling_map)))

full_annotations = (
    full_annotations
    .with_columns(
        pl.col("annotation_version")
        .eq("new_annotations")
        .cast(pl.Int8)
        .alias("_is_new")
    )
    .sort(["DFCI_MRN", "_is_new"], descending=[False, True])
    .unique(subset="DFCI_MRN", keep="first", maintain_order=True)
    .drop("_is_new")
)

full_annotations = full_annotations.with_columns(
    pl.col('manual_annotation').replace({'non_prostate_primary' : 'conventional', 'biomarker' : 'conventional'})
)

joined_labels = llm_labels.join(full_annotations, on='DFCI_MRN', how='left').filter(pl.col('manual_annotation').is_not_null())

cols_to_keep = ['DFCI_MRN', 'primary_label', 'manual_annotation', 'annotation_version', 'nepc_review_status', 'nepc_status', 'nepc_evidence_type', 
                'nepc_quote', 'nepc_source_note_date', 'stated_nepc_dx_quote',  'avpc_review_status', 'is_metastatic', 'metastatic_quote', 
                'is_castration_resistant', 'crpc_quote', 'is_mcrpc', 'c1_met', 'c1_quote', 'c1_date', 'c1_source_note_date', 'c2_met', 'c2_quote', 
                'c2_date', 'c2_source_note_date', 'c3_met', 'c3_quote', 'c3_date', 'c3_source_note_date', 'c4_met', 'c4_quote', 'c4_date', 'c4_source_note_date', 
                'c5_met', 'c5_quote', 'c5_date', 'c5_source_note_date', 'c6_met', 'c6_quote', 'c6_date', 'c6_source_note_date', 'c7_met', 'c7_quote', 
                'c7_date', 'c7_source_note_date', 'c4_size_quote', 'c4_gleason_quote', 'c5_psa_quote', 'c5_bone_quote', 'c6_marker_quote', 'c6_lab_quote', 
                'c7_adt_start_quote', 'c7_crpc_quote', 'c7_interval_days', 'avpc_criteria', 'n_avpc_criteria', 'is_avpc', 'avpc_date']
labels_to_write = joined_labels[cols_to_keep]

df = labels_to_write.with_columns([
    pl.col(col).struct.json_encode().alias(col)
    if isinstance(dtype, pl.Struct)
    else pl.col(col).list.join(", ").alias(col)
    if isinstance(dtype, pl.List) and dtype.inner == pl.String
    else pl.col(col)
    for col, dtype in labels_to_write.schema.items()
])
# df.write_csv(COMPASS_PATH / 'joined_llm_labels.tsv', separator='\t')
# review_df = pl.read_csv(COMPASS_PATH / 'patrick_review.tsv', separator='\t')
# review_df.filter(pl.col('annotation_version') == 'new_annotations').write_csv(COMPASS_PATH / 'patrick_review_new_annootations.tsv', separator='\t')
review_df = pl.read_csv(COMPASS_PATH / 'patrick_review.tsv', separator='\t')

import polars as pl

review_df = review_df.with_columns(
    # Low confidence: use patrick_review if available
    pl.coalesce(
        pl.col("patrick_review"),
        pl.col("manual_annotation")
    ).alias("updated_manual_low_confidence"),

    # High confidence: retain manual_annotation if low certainty
    pl.when(
        pl.col("updated_manual")
        .str.contains("low certainty", literal=True)
        .fill_null(False)
    )
    .then(pl.col("manual_annotation"))
    .otherwise(
        pl.coalesce(
            pl.col("patrick_review"),
            pl.col("manual_annotation")
        )
    )
    .alias("updated_manual_high_confidence")
)
review_df[['DFCI_MRN', 'updated_manual_low_confidence', 'updated_manual_high_confidence']].write_csv('/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/LLM_NEPC_labels/full_manual_annotations.csv')